# Experiment 1: Population Creation and Encoding Methods
**Aim:** To create and represent a population of individuals using different encoding methods in Python.

**Theory:** In a Genetic Algorithm (GA) each candidate solution is an *individual* (chromosome) made of *genes*. How a solution is encoded decides which crossover/mutation operators can be used. Common encodings:
- **Binary** – genes are 0/1 (knapsack, function optimisation)
- **Integer** – genes are integers in a range (scheduling, resource allocation)
- **Real-valued** – genes are floats (continuous optimisation)
- **Permutation** – genes are a reordering of items (TSP)
- **Gray code** – binary code where adjacent values differ by one bit

In [1]:
import random
import numpy as np
import pandas as pd

random.seed(42)
np.random.seed(42)

## 1. Binary encoding

In [2]:
def init_binary(pop_size, n_bits):
    return [[random.randint(0, 1) for _ in range(n_bits)] for _ in range(pop_size)]

binary_pop = init_binary(5, 8)
for i, ind in enumerate(binary_pop):
    print(f"Individual {i}: {ind}")

Individual 0: [0, 0, 1, 0, 0, 0, 0, 0]
Individual 1: [1, 0, 0, 0, 0, 0, 0, 0]
Individual 2: [1, 0, 1, 1, 0, 0, 1, 1]
Individual 3: [1, 0, 0, 1, 0, 0, 1, 0]
Individual 4: [1, 1, 1, 0, 1, 0, 1, 0]


## 2. Integer encoding

In [3]:
def init_integer(pop_size, n_genes, low, high):
    return [[random.randint(low, high) for _ in range(n_genes)] for _ in range(pop_size)]

integer_pop = init_integer(5, 6, 1, 9)
for i, ind in enumerate(integer_pop):
    print(f"Individual {i}: {ind}")

Individual 0: [9, 5, 6, 4, 2, 1]
Individual 1: [4, 5, 2, 4, 2, 7]
Individual 2: [5, 8, 6, 3, 6, 6]
Individual 3: [4, 5, 2, 3, 9, 4]
Individual 4: [3, 8, 7, 5, 9, 4]


## 3. Real-valued encoding

In [4]:
def init_real(pop_size, n_genes, low, high):
    return [[round(random.uniform(low, high), 3) for _ in range(n_genes)] for _ in range(pop_size)]

real_pop = init_real(5, 4, -5.0, 5.0)
for i, ind in enumerate(real_pop):
    print(f"Individual {i}: {ind}")

Individual 0: [1.846, 3.429, 2.76, -2.71]
Individual 1: [-4.679, -1.845, -2.323, -2.89]
Individual 2: [4.429, 3.764, -1.853, 1.554]
Individual 3: [-1.044, 4.145, -0.411, -2.351]
Individual 4: [-2.534, 0.614, -2.373, 0.846]


## 4. Permutation encoding (e.g., a tour through 8 cities)

In [5]:
def init_permutation(pop_size, n_items):
    pop = []
    for _ in range(pop_size):
        ind = list(range(n_items))
        random.shuffle(ind)
        pop.append(ind)
    return pop

perm_pop = init_permutation(5, 8)
for i, ind in enumerate(perm_pop):
    print(f"Individual {i}: {ind}  (valid permutation: {sorted(ind) == list(range(8))})")

Individual 0: [4, 7, 0, 3, 5, 1, 2, 6]  (valid permutation: True)
Individual 1: [4, 0, 2, 3, 6, 5, 7, 1]  (valid permutation: True)
Individual 2: [1, 0, 5, 2, 7, 4, 3, 6]  (valid permutation: True)
Individual 3: [3, 7, 0, 6, 2, 4, 5, 1]  (valid permutation: True)
Individual 4: [7, 4, 5, 2, 0, 3, 1, 6]  (valid permutation: True)


## 5. Gray code encoding and decoding

In [6]:
def binary_to_gray(bits):
    gray = [bits[0]]
    for i in range(1, len(bits)):
        gray.append(bits[i - 1] ^ bits[i])
    return gray

def gray_to_binary(gray):
    bits = [gray[0]]
    for i in range(1, len(gray)):
        bits.append(bits[i - 1] ^ gray[i])
    return bits

for n in range(8):
    b = [int(x) for x in format(n, '04b')]
    g = binary_to_gray(b)
    print(n, "binary:", b, "gray:", g, "back:", gray_to_binary(g) == b)

0 binary: [0, 0, 0, 0] gray: [0, 0, 0, 0] back: True
1 binary: [0, 0, 0, 1] gray: [0, 0, 0, 1] back: True
2 binary: [0, 0, 1, 0] gray: [0, 0, 1, 1] back: True
3 binary: [0, 0, 1, 1] gray: [0, 0, 1, 0] back: True
4 binary: [0, 1, 0, 0] gray: [0, 1, 1, 0] back: True
5 binary: [0, 1, 0, 1] gray: [0, 1, 1, 1] back: True
6 binary: [0, 1, 1, 0] gray: [0, 1, 0, 1] back: True
7 binary: [0, 1, 1, 1] gray: [0, 1, 0, 0] back: True


## 6. Decoding: mapping a binary chromosome to a real number in [a, b]
$x = a + \frac{\text{decimal}(bits)}{2^{n}-1}(b-a)$

In [7]:
def decode_binary(bits, low, high):
    value = int("".join(map(str, bits)), 2)
    return low + value * (high - low) / (2 ** len(bits) - 1)

pop = init_binary(5, 10)
rows = [{"chromosome": "".join(map(str, ind)),
         "decimal": int("".join(map(str, ind)), 2),
         "x in [-5, 5]": round(decode_binary(ind, -5, 5), 4)} for ind in pop]
pd.DataFrame(rows)

,chromosome,decimal,"x in [-5, 5]"
0,0100100110,294,-2.1261
1,0110000010,386,-1.2268
2,0010110011,179,-3.2502
3,1110000100,900,3.7977
4,0000000100,4,-4.9609


## Conclusion
Binary, integer, real-valued, permutation and Gray encodings were created and decoded. The choice of encoding depends on the problem: binary/Gray for bit-strings or discretised numbers, real-valued for continuous problems, and permutation for ordering problems.